# Formula E — Video Lab (Challenge 3: the Race Control Observer)

This notebook is where you **develop** the Video Verifier — the piece that reads the
trackside CCTV and answers one question about a telemetry-flagged stop: *is the racing
line still blocked, or did it clear?* It's a fast loop (seconds per try, versus minutes
to rebuild a Cloud Run service), and the prompt you land here is the one you'll drop into
`starter/video_verifier/verifier.py`.

What you'll do:
1. **Explore the telemetry** and find a stopped car yourself.
2. **Dissect the mosaics** — 24 cameras tiled into six 2×2 grids: **24 cameras → 6 calls.**
3. **Prove the time alignment** — that mp4 offset *N* really is race-second *N*.
4. **Make your first Gemini call** on a `gs://` video *slice* — no download, no ffmpeg.
5. **Write the persistence prompt yourself** — this is Task 2, the heart of the hack.

> **As you go: READ each cell before you run it.** The point is to understand the pieces,
> not just execute them — you're about to rebuild this logic in `verifier.py`.

## 0 · Setup — read it, then run it

In [ ]:
import os, json, gzip, time, subprocess, urllib.request
from datetime import datetime, timedelta, timezone

# --- Project auto-detects (env -> ADC -> gcloud). Override by setting it here. ---
PROJECT_ID = ""
if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "") or ""
if not PROJECT_ID:
    try:
        import google.auth; PROJECT_ID = google.auth.default()[1] or ""
    except Exception: pass
if not PROJECT_ID:
    try:
        PROJECT_ID = subprocess.run(["gcloud","config","get-value","project"],
                                    capture_output=True, text=True).stdout.strip()
    except Exception: pass
assert PROJECT_ID, "Could not auto-detect PROJECT_ID - set it manually above."

MOSAICS_BASE = f"gs://{PROJECT_ID}-fe-mosaics/mosaics"   # your project's staged mosaics
GEM_MODEL = "gemini-3.5-flash"
GUNTHER_GROUP = "grp_02_cam05_cam06_cam07_cam08"          # holds Cam05 (panel TL) — the Günther stop

# The bundled Berlin R10 telemetry (1 Hz). Pull the one file from the repo.
FRAMES = "frames.jsonl.gz"
RAW = "https://raw.githubusercontent.com/haggman/formula-e-race-control-observer/main/simulator/src/frames.jsonl.gz"
if not os.path.exists(FRAMES):
    urllib.request.urlretrieve(RAW, FRAMES)

# Green flag: race-second 0 = 2024-05-12 13:04:00 UTC (= 15:04:00 Berlin, CEST).
GREEN_UTC = datetime(2024, 5, 12, 13, 4, 0, tzinfo=timezone.utc)
print("project:", PROJECT_ID, "| mosaics:", MOSAICS_BASE, "| model:", GEM_MODEL)

## 1 · Explore the telemetry — find the stop yourself

Load the frames and plot car **#7 (Günther)**'s speed. Don't take the incident on faith —
*see* the moment the car stops. (Read the cell, then run it.)

In [ ]:
import matplotlib.pyplot as plt

series, drivers = {}, {}
with gzip.open(FRAMES, "rt") as f:
    for line in f:
        o = json.loads(line); t = o["race_time_s"]
        for cc in o["cars"]:
            series.setdefault(cc["car_number"], {})[t] = cc
            drivers[cc["car_number"]] = cc["driver_name"]

def speed(n, a, b):
    ts = [t for t in range(a, b+1) if t in series[n]]
    return ts, [series[n][t]["speed_kmh"] for t in ts]

ts, sp = speed(7, 600, 820)
plt.figure(figsize=(10,3))
plt.plot(ts, sp); plt.axhline(5, ls="--", c="grey", lw=1)
plt.title(f"#7 {drivers[7]} — speed (km/h)"); plt.xlabel("race-second"); plt.ylabel("km/h")
plt.axvspan(693, 789, color="red", alpha=0.08); plt.tight_layout(); plt.show()

def find_stop(n, thresh=5.0, hold=6):     # the detector's rule: <=5 km/h held >= 6 s
    run = []
    for t in sorted(series[n]):
        if series[n][t]["speed_kmh"] <= thresh:
            run.append(t)
            if run[-1]-run[0] >= hold: return run[0]
        else: run = []
    return None
print("first sustained stop for #7 at race-second:", find_stop(7))

### A stopped car isn't the hard part — a stopped car *out on the track* is

Plot car **#33 (Ticktum)** next to #7. Both drop to zero. But telemetry is **not** blind
to the difference: it knows #33 is in the **pit lane** from its GPS, and the board says so
out loud — *"routine pit stop, not a track incident."* No camera needed for that call.

The camera exists for the *other* case — a car stopped **on the racing surface**, like #7.
There, telemetry sees the stop but can't judge what it *means*: has the car really come to
rest, is the line actually blocked, and is it **still** blocked a minute later? That's the
question you'll teach Gemini to answer.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3), sharey=True)
for a, (n, lbl) in zip(ax, [(7, "#7 Günther — STOPPED ON TRACK"), (33, "#33 Ticktum — PIT LANE")]):
    t0 = find_stop(n); ts, sp = speed(n, t0-15, t0+40)
    a.plot(ts, sp); a.axhline(5, ls="--", c="grey", lw=1); a.set_title(lbl); a.set_xlabel("race-second")
ax[0].set_ylabel("km/h"); plt.tight_layout(); plt.show()
print("Both hit 0. Telemetry flags #33 as a pit stop by its GPS and moves on.")
print("#7 is the one that needs a second opinion: is the racing line really, still blocked?")

## 2 · Dissect the mosaics — 24 cameras become 6 Gemini calls

The circuit has 24 CCTV cameras. Asking Gemini about each, per incident, is 24 calls.
Instead they're pre-tiled into **six 2×2 mosaics** — four adjacent cameras per file in a
fixed layout: **TL, TR, BL, BR**. One call covers four cameras. **24 → 6.** The model
tells you which *panel* it saw the incident in; the code maps that panel back to a camera.

In [ ]:
GROUPS = {
    "grp_01_cam01_cam02_cam03_cam04": ["Cam01","Cam02","Cam03","Cam04"],
    "grp_02_cam05_cam06_cam07_cam08": ["Cam05","Cam06","Cam07","Cam08"],
    "grp_03_cam09_cam10_cam11_cam12": ["Cam09","Cam10","Cam11","Cam12"],
    "grp_04_cam13_cam14_cam15_cam16": ["Cam13","Cam14","Cam15","Cam16"],
    "grp_05_cam17_cam18_cam19_cam20": ["Cam17","Cam18","Cam19","Cam20"],
    "grp_06_cam21_cam22_cam23_cam24": ["Cam21","Cam22","Cam23","Cam24"],
}
PANELS = ["TL","TR","BL","BR"]
print(f"{len(GROUPS)} groups x 4 cameras = 24 cameras, but only {len(GROUPS)} Gemini calls per incident.")

Pull one mosaic into the runtime and look at a single frame — see the 2×2 grid, and the clock burned into each panel.

In [ ]:
try:
    import cv2
except Exception:
    subprocess.run(["pip","install","-q","opencv-python-headless"], check=True); import cv2

def ensure_local(gid):
    p = f"{gid}.mp4"
    if not os.path.exists(p):
        subprocess.run(["gcloud","storage","cp",f"{MOSAICS_BASE}/{gid}.mp4",p], check=True)
    return p

def frame_at(gid, offset_s):
    cap = cv2.VideoCapture(ensure_local(gid)); fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(offset_s*fps)); ok, img = cap.read(); cap.release()
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if ok else None

gid = "grp_01_cam01_cam02_cam03_cam04"
plt.figure(figsize=(8,6)); plt.imshow(frame_at(gid, 693)); plt.axis("off")
plt.title(f"{gid} @ 693s — panels TL/TR/BL/BR = {GROUPS[gid]}"); plt.show()

## 3 · Prove the time alignment — everything rests on this

The mosaics are **1 FPS starting at race-second 0**, so **mp4 offset N = race-second N**.
That's what lets you ask for "the footage around the stop" with no clock conversion. Prove
it: at offset **693**, the burned-in wall-clock should read about **13:15:33 UTC** (green
flag 13:04:00 + 693 s) = **15:15:33 Berlin**. The four panels are separate cameras, so
their clocks can differ by a second or two — anything within a couple of seconds of
13:15:33 is a match.

In [ ]:
want = GREEN_UTC + timedelta(seconds=693)
print(f"offset 693s should show ~{want:%H:%M:%S} UTC (±a second or two)  =  ~{want + timedelta(hours=2):%H:%M:%S} Berlin")
plt.figure(figsize=(8,6)); plt.imshow(frame_at(GUNTHER_GROUP, 693)); plt.axis("off")
plt.title("Read the clocks — within a second or two of 13:15:33?"); plt.show()

## 4 · Your first Gemini call — hand it a `gs://` slice

The novel move: point Gemini **straight at the mosaic in the bucket** and pass
`videoMetadata` start/end offsets, so it decodes **only** the window. No download, no
ffmpeg. Start open-ended — just ask what it sees around the stop.

### Docs worth a look
- Video understanding + clipping with `videoMetadata` offsets: https://ai.google.dev/gemini-api/docs/video-understanding
- The `google-genai` SDK types (`Part`, `FileData`, `VideoMetadata`): https://googleapis.github.io/python-genai/

In [ ]:
try:
    from google import genai
    from google.genai import types
except Exception:
    subprocess.run(["pip","install","-q","google-genai"], check=True)
    from google import genai
    from google.genai import types
import google.auth
GEM = genai.Client(vertexai=True, project=(PROJECT_ID or google.auth.default()[1]), location="global")

def gemini_slice(group_id, t, lead=10, tail=50, prompt="What do you see in this CCTV clip? Is any car stopped on the track by the end?"):
    start, end = max(0, t-lead), t+tail
    vpart = types.Part(file_data=types.FileData(file_uri=f"{MOSAICS_BASE}/{group_id}.mp4", mime_type="video/mp4"),
                       video_metadata=types.VideoMetadata(start_offset=f"{start}s", end_offset=f"{end}s"))
    resp = GEM.models.generate_content(model=GEM_MODEL,
        contents=[types.Content(role="user", parts=[vpart, types.Part(text=prompt)])],
        config=types.GenerateContentConfig(temperature=0.2))
    return resp.text

print(gemini_slice(GUNTHER_GROUP, 693))   # the group with Cam05, where Günther is visible

## 5 · Write the persistence prompt — this is Task 2, and it's yours to invent

The heart of the hack. The rest of the code just needs Gemini to answer, in JSON, **one**
question about the track. Your job is to *pose* that question well.

**What the code needs back** (these keys):
`blockage` (bool), `cleared` (bool), `panel` (`TL|TR|BL|BR|none`), `feed_live` (bool),
`seen_car` (number or null), `what_you_see` (string), `confidence` (0..1).

**An example of a good answer** (for the Günther stop):
```json
{"blockage": true, "cleared": false, "panel": "TL", "feed_live": true, "seen_car": 7,
 "what_you_see": "A dark blue car is stopped against the wall in the top-left camera and is still there at the end of the clip while others pass.", "confidence": 0.9}
```

**Your job:** pose the question so that **one** answer separates a real retirement from a
car that spun and drove away — harder than *"is there a stopped car?"* The wording is the
whole hack; think about *when* in the clip you're judging. We prepend the clip/panel
context and the JSON request for you, so `PROMPT` is just your question. **Try it yourself
first** — the hints and full solution are in section 6 if you stall. Edit `PROMPT` and rerun.

In [ ]:
# >>> WRITE YOUR PROMPT HERE <<<   (just the question; we add the JSON contract + panel map)
# Goal: get Gemini to return the example JSON above for the Günther stop.
# Try it on your own first. Stuck? Section 6 has a half-written hint and a full solution.
PROMPT = '''
YOUR PROMPT HERE
'''

_JSON_ASK = ('Return ONLY a JSON object: {"blockage": bool, "cleared": bool, '
             '"panel": "TL|TR|BL|BR|none", "feed_live": bool, "seen_car": <number or null>, '
             '"what_you_see": string, "confidence": number}')

def _parse(text):
    s = (text or "").strip(); a, b = s.find("{"), s.rfind("}")
    if a != -1 and b > a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    return {}

def _truthy(v):                      # tolerant: accept True / "true" / "yes" / 1
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return v != 0
    if isinstance(v, str): return v.strip().lower() in ("true","yes","y","1","blocked","blockage")
    return False

def _call(group_id, t, prompt, lead=10, tail=50):
    start, end = max(0, t-lead), t+tail
    cams = GROUPS[group_id]
    panelmap = f"Panels map to cameras: TL={cams[0]}, TR={cams[1]}, BL={cams[2]}, BR={cams[3]}."
    full = f"{prompt}\nThis is a ~{end-start}s CCTV clip: a 2x2 mosaic. {panelmap}\n{_JSON_ASK}"
    vpart = types.Part(file_data=types.FileData(file_uri=f"{MOSAICS_BASE}/{group_id}.mp4", mime_type="video/mp4"),
                       video_metadata=types.VideoMetadata(start_offset=f"{start}s", end_offset=f"{end}s"))
    resp = GEM.models.generate_content(model=GEM_MODEL,
        contents=[types.Content(role="user", parts=[vpart, types.Part(text=full)])],
        config=types.GenerateContentConfig(temperature=0.2, response_mime_type="application/json"))
    return _parse(resp.text)

print("PROMPT set. Test it on the single camera below.")

### Step 1 — get it right on ONE camera you can see

`grp_02_cam05_cam06_cam07_cam08` holds **Cam05 (panel TL)**, where the Günther stop is
visible — you saw the car in the frame in section 3. Iterate `PROMPT` above until this one
camera returns `blockage: true` with a sensible description. Small loop, fast feedback.

In [ ]:
def run_one(group_id, t, prompt):
    plt.figure(figsize=(6,4.5)); plt.imshow(frame_at(group_id, t)); plt.axis("off")
    plt.title(f"{group_id} @ {t}s"); plt.show()
    d = _call(group_id, t, prompt)
    print(json.dumps(d, indent=2))
    print("=> blockage:", _truthy(d.get("blockage")), "| cleared:", _truthy(d.get("cleared")))
    return d

_ = run_one(GUNTHER_GROUP, 693, PROMPT)   # want: blockage True, panel TL, a blue car by the wall

### Step 2 — sweep all six, fuse to one verdict

Now run *your same prompt* across all six mosaics and combine them into a single verdict —
exactly what `_aggregate` will do in `verifier.py`: any blockage → **blocked**; else any
recovery → **cleared**; else **unseen**.

In [ ]:
def sweep(t, prompt, lead=10, tail=50):
    print(f"@ t={t}s — sweeping {len(GROUPS)} groups")
    blocked, cleared = [], []
    for gid, cams in GROUPS.items():
        d = _call(gid, t, prompt, lead, tail)
        panel = str(d.get("panel", "none"))
        cam = cams[PANELS.index(panel)] if panel in PANELS else None
        st = "BLOCKED" if _truthy(d.get("blockage")) else ("cleared" if _truthy(d.get("cleared")) else "-")
        print(f"  {gid:32} {st:8} panel={panel:5} cam={cam} conf={d.get('confidence','?')}")
        if _truthy(d.get("blockage")): blocked.append((cam, d))
        elif _truthy(d.get("cleared")): cleared.append((cam, d))
    if blocked:
        cam, d = max(blocked, key=lambda x: x[1].get("confidence", 0) or 0)
        print(f"  VERDICT: BLOCKED on {cam} — {d.get('what_you_see','')}")
        return "blocked"
    if cleared:
        print("  VERDICT: cleared (a car was there but recovered / line clear)")
        return "cleared"
    print("  VERDICT: unseen (no group reported a blockage)")
    return "unseen"

_ = sweep(693, PROMPT)   # Günther -> want BLOCKED on Cam05

### Step 3 — the verification passes

Three known cases. With a good prompt you want **blocked / cleared / blocked**:
- **693** — Günther #7 retires → blocked (Cam05)
- **1373** — Evans #9 has a big moment but drives away → cleared
- **1780** — #48 stops then recovers while #23 is still stranded on Cam07 → blocked

If Günther comes back `unseen`, your prompt is too timid or it's judging a single moment
instead of the end of the window — go back to Step 1 and tune it on the one camera.

In [ ]:
for t in (693, 1373, 1780):
    sweep(t, PROMPT)

## 6 · Stuck on the wording? Two reference prompts

Reach for the **half** one first — it leaves the key sentence for you to finish. If you
just want to move on, set `PROMPT = SOLUTION_PROMPT_FULL` and rerun section 5. Using it is
shipping, not cheating.

In [ ]:
SOLUTION_PROMPT_HALF = '''
You are a race-control video verifier deciding whether a SAFETY CAR is warranted.
Telemetry flagged a car that may be stopped on the track. Watch the whole clip.
TODO: tell the model to judge the TRACK STATE *by the END* of the clip, and to separate
a car that is STILL blocking the racing line (blockage=true) from one that DROVE AWAY /
was recovered so the line is clear (cleared=true). Remember a car can be stopped early
and gone by the end.
<< finish the wording so ONE answer separates a retirement from a spin-and-recover >>
'''

SOLUTION_PROMPT_FULL = '''
You are a race-control video verifier deciding whether a SAFETY CAR is warranted.
Telemetry flagged a car that may be stopped on the track. Watch the whole clip and judge
the TRACK STATE by the END of it (the safety call is about the track, not which car):
- A car STILL stopped/stranded on or beside the racing line at the end (a persistent
  obstruction, maybe with marshals or a recovery vehicle): blockage=true, cleared=false.
- A car appeared but DROVE AWAY / was recovered / the line is clear by the end:
  blockage=false, cleared=true.
- No stopped car at any point: blockage=false, cleared=false.
Base the answer on the END of the window, not a single moment. Note whether other cars
are still moving (feed live).
'''

# To use one:  PROMPT = SOLUTION_PROMPT_FULL   (then re-run the Step 1/2/3 cells)
print("Reference prompts loaded. Try the HALF one first.")

## 7 · Port it — take your prompt to `verifier.py`

You've got a prompt that works. Move it into the real component so the correlator can call
it. In `starter/video_verifier/verifier.py`, fill the three stubs:

- **`_prompt(...)`** — the persistence question + JSON contract you just tuned (the stub
  shows where the panel/camera line goes).
- **`VideoVerifier._verify_group(...)`** — one Gemini call over one `gs://` slice (the
  video Part + `videoMetadata` offsets from section 4), returning the parsed dict.
- **`VideoVerifier._aggregate(...)`** — fuse the six replies into one `VideoVerdict`
  (core: `blocked` vs `unseen`; the honest `cleared` and `error` states are Bonuses 4 & 5).

Then test it standalone, no full stack required:

```bash
python -m starter.video_verifier.verifier --at 693 --cars 7        # -> blocked  (Günther)
python -m starter.video_verifier.verifier --at 1698 --cars 17 23   # -> blocked  (Fenestraz + Nato)
python -m starter.video_verifier.verifier --at 1780 --cars 48      # -> blocked
```

When those pass, restart your correlator (`python -m correlator.service`), click the
**Günther** button in the console, and watch the recommendation go from **DOUBLE YELLOW**
to **SAFETY CAR · corroborated**. That board lighting up is *your* code authorising the flag.